<p align="center">
  <img src="https://img.shields.io/badge/BINX%20TECH-AI%20%26%20ML%20INTERNSHIP-0B1F3A?style=for-the-badge" alt="BinX Tech"/>
</p>

<h1 align="center">🚀 Week 9 · Day 1</h1>
<h3 align="center">Sprint 4 Planning, Model Serialization &amp; MLOps</h3>

<p align="center">
  <img src="https://img.shields.io/badge/PHASE-3%20CAPSTONE-0B1F3A?style=for-the-badge" alt="Phase 3"/>
  <img src="https://img.shields.io/badge/SPRINT-4%20OF%204-1B4F8C?style=for-the-badge" alt="Sprint 4"/>
  <img src="https://img.shields.io/badge/DAY-1%20OF%205-0096C7?style=for-the-badge&logoColor=black" alt="Day 1 of 5"/>
  <img src="https://img.shields.io/badge/TOPIC-DEPLOYMENT%20PREP-0077B6?style=for-the-badge" alt="Deployment Prep"/>
  <img src="https://img.shields.io/badge/CAPSTONE-CARDIAC%20MONITORING-03045E?style=for-the-badge" alt="Cardiac Monitoring Capstone"/>
</p>

<p align="center"><i>"A model in a notebook helps no one. Deployment is what turns a trained model into something real users can actually use."</i></p>

---

## 📖 The Story So Far

Sprint 3 (Week 8) closed with an honest verdict: **Logistic Regression is the pragmatic choice** for the Cardiac
Monitoring capstone — Random Forest barely beat it (0.886 vs. 0.886 accuracy, 0.930 vs. 0.930 AUC) and only earned
its keep in that notebook because `TreeExplainer` made SHAP convenient, not because it was actually better. Sprint 4
picks up that retrospective directly: **this sprint ships the simple model, not the fancier one that didn't earn its
complexity.**

Today opens the final sprint of the Phase 3 capstone — the one where the model stops living in a notebook and
becomes a real, callable service. Day 1's job is everything that has to happen *before* a single line of serving
code is written: plan the sprint, save the trained model and its preprocessing to disk in a form another program can
load, and lock down the environment so today's result is reproducible tomorrow, next week, or on a teammate's
machine.

## 🎯 Learning Objectives

| | Objective |
|---|---|
| 🗂️ | Complete Sprint 4 planning and define the deployment backlog |
| 💾 | Serialize the trained model *and* its preprocessing objects for production |
| 🔁 | Apply reproducibility practices (pinned requirements, MLflow tracking, fixed seeds) ahead of deployment |


## 1&nbsp;·&nbsp;Sprint 4 Planning

**Sprint goal:** deploy the Cardiac Monitoring model as a live, public application — not just a notebook that runs
locally.

**Backlog (this sprint, five days):**

| Day | Backlog Item |
|---|---|
| 1 (today) | Serialize the trained model + preprocessing; freeze the deployment environment |
| 2 | Serve the model behind a FastAPI `/predict` endpoint with request validation |
| 3 | Build an interactive Streamlit dashboard for non-technical users |
| 4 | Deploy publicly (Hugging Face Spaces / Render / Railway) |
| 5 | Repository polish to the Definition of Done; Sprint Review + full-project Retrospective |

**Carried forward from the Sprint 3 retrospective:** ship **Logistic Regression**, not Random Forest. Sprint 3's own
honesty check showed the extra complexity never paid for itself — Sprint 4 is where that conclusion actually gets
acted on instead of just written down.

## 2&nbsp;·&nbsp;Why Deployment Matters

A model in a notebook helps no one — it can't be queried by a website, a mobile app, or another backend. Deployment
is what turns four sprints of careful data work into something a real user (or a real doctor, for this capstone's
theme) can actually get a prediction from. It is also the single feature that separates a portfolio project an
employer takes seriously from a purely academic exercise finished at "the model works in my notebook."


## 3&nbsp;·&nbsp;Rebuilding the Model That Ships

Before anything can be serialized, the exact pipeline that will be deployed needs to be trained one more time, in
one place, so the artifacts saved to disk match precisely what Sprint 3 validated — same data, same split, same
random seed, same model. This is **Logistic Regression only** — the sprint's own retrospective ruled out Random
Forest.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("heart.csv")
print(f"Loaded {len(df)} patients, {df.shape[1]} columns")
df.head()

Loaded 918 patients, 12 columns


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

X = df.drop(columns=["HeartDisease"])
y = df["HeartDisease"]

cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()

RANDOM_SEED = 42  # fixed for reproducibility -- see Section 6

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), cat_cols),
])

# Fit preprocessing and model as two separate objects (not one Pipeline) --
# this is deliberate: Day 2's serving API needs to load and inspect them independently.
X_train_pre = preprocessor.fit_transform(X_train)
X_test_pre = preprocessor.transform(X_test)

model = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)
model.fit(X_train_pre, y_train)

pred = model.predict(X_test_pre)
proba = model.predict_proba(X_test_pre)[:, 1]

acc = accuracy_score(y_test, pred)
auc = roc_auc_score(y_test, proba)
print(f"Logistic Regression -- Accuracy: {acc:.4f}   ROC-AUC: {auc:.4f}")
print("Matches Sprint 3's Day 5 result -- same data, split, seed, and model.")

Logistic Regression -- Accuracy: 0.8859   ROC-AUC: 0.9297
Matches Sprint 3's Day 5 result -- same data, split, seed, and model.


## 4&nbsp;·&nbsp;Model Serialization

Before a separate application (the FastAPI service built tomorrow) can use this model, it has to be saved to disk in
a form that can be loaded **without retraining**. The method depends on what's being saved:

| Object | Save Method | File |
|---|---|---|
| Scikit-learn classifier | `joblib.dump()` | `model.joblib` |
| Preprocessing (`ColumnTransformer`) | `joblib.dump()` | `preprocessor.joblib` |

> [!WARNING]
> **The training/serving skew rule (from Week 8):** the preprocessing object must be saved *alongside* the model,
> not re-implemented by hand in the serving code. Loading only `model.joblib` and manually rewriting the scaling and
> encoding logic in the API is a top real-world deployment bug — any tiny mismatch (a different encoding order, a
> forgotten `handle_unknown="ignore"`) silently produces wrong predictions in production while looking fine in
> testing.

In [3]:
import joblib

joblib.dump(model, "model.joblib")
joblib.dump(preprocessor, "preprocessor.joblib")

import os
print("Saved artifacts:")
for f in ["model.joblib", "preprocessor.joblib"]:
    print(f"  {f:<22} {os.path.getsize(f):>8,} bytes")

Saved artifacts:
  model.joblib              1,007 bytes
  preprocessor.joblib       4,118 bytes


## 5&nbsp;·&nbsp;Verifying the Round Trip

Saving artifacts is only half the job -- the hands-on lab explicitly calls for **loading them back and reproducing a
known prediction** before trusting them in a deployment. This is the cheapest possible test that catches a broken
serialization (a version mismatch, a corrupted file, a preprocessing object that didn't actually fit) before it
becomes a production incident.

In [4]:
# Load the artifacts back exactly as a fresh process (e.g. the FastAPI app) would.
loaded_model = joblib.load("model.joblib")
loaded_preprocessor = joblib.load("preprocessor.joblib")

# Reproduce a known prediction: patient #0 from the held-out test set.
known_patient = X_test.iloc[[0]]
known_actual = y_test.iloc[0]

original_pred = model.predict(preprocessor.transform(known_patient))[0]
reloaded_pred = loaded_model.predict(loaded_preprocessor.transform(known_patient))[0]

print(f"Original in-memory prediction:  {original_pred}")
print(f"Reloaded-from-disk prediction:  {reloaded_pred}")
print(f"Actual label:                   {known_actual}")

assert original_pred == reloaded_pred, "Round-trip mismatch -- serialization is broken!"
print("\\nRound-trip verified: the reloaded artifacts reproduce the exact same prediction.")

Original in-memory prediction:  1
Reloaded-from-disk prediction:  1
Actual label:                   1
\nRound-trip verified: the reloaded artifacts reproduce the exact same prediction.


> [!NOTE]
> This assertion is the whole point of the exercise: it is a two-line automated check that the artifacts Day 2's
> API will load are trustworthy, run *today* rather than discovered as a bug during tomorrow's demo.

## 6&nbsp;·&nbsp;MLOps &amp; Reproducibility Recap

Three practices, all already used earlier in this internship, matter most right before a deployment:

| Practice | Where It Shows Up Here |
|---|---|
| **Fixed random seeds** | `random_state=42` used identically in the train/test split and the model — the same as every prior sprint |
| **Experiment tracking (MLflow)** | Every model version logged with its parameters and metrics, so today's shipped model is traceable later |
| **Pinned `requirements.txt`** | The deployment environment installs the *exact* library versions used here — not just "the latest scikit-learn" |

Without these three, a model that works today in this notebook can behave differently — or fail to install
entirely — in tomorrow's deployment environment.

In [5]:
import mlflow
import mlflow.sklearn

mlflow.set_experiment("cardiac-monitoring-capstone")

with mlflow.start_run(run_name="week9-day1-logreg-deployment-candidate"):
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("random_state", RANDOM_SEED)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("n_features", X.shape[1])

    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("roc_auc", auc)

    mlflow.sklearn.log_model(model, name="model")

    run_id = mlflow.active_run().info.run_id
    print(f"Logged MLflow run: {run_id}")
    print(f"  accuracy = {acc:.4f}")
    print(f"  roc_auc  = {auc:.4f}")

print("\\nEvery future model version can now be logged the same way -- run_id makes today's exact")
print("model, its parameters, and its metrics traceable months from now.")

2026/09/14 19:04:25 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /usr/local/lib/python3.11/dist-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


2026/09/14 19:04:26 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/09/14 19:04:26 INFO mlflow.store.db.utils: Updating database tables


2026/09/14 19:04:27 INFO mlflow.tracking.fluent: Experiment with name 'cardiac-monitoring-capstone' does not exist. Creating a new experiment.


Logged MLflow run: f95a93680ee44217ba788f2e9c188c3f
  accuracy = 0.8859
  roc_auc  = 0.9297
\nEvery future model version can now be logged the same way -- run_id makes today's exact
model, its parameters, and its metrics traceable months from now.


> [!NOTE]
> **Why this over a filename convention:** naming files `model_v2_final_FINAL.joblib` doesn't record *what changed*
> between versions. An MLflow run records the parameters, the metrics, and the artifact together, so "was this the
> run with `max_iter=2000` or the one before the seed was fixed?" always has an exact answer.

In [6]:
pinned_requirements = """\
# Deployment environment for the Cardiac Monitoring capstone (Week 9 Sprint 4)
# Frozen on Day 1 so Day 2's serving environment installs these exact versions.
pandas==""" + pd.__version__ + r"""
numpy==""" + np.__version__ + r"""
scikit-learn==""" + __import__("sklearn").__version__ + r"""
joblib==""" + joblib.__version__ + r"""
fastapi==""" + __import__("fastapi").__version__ + r"""
uvicorn==""" + __import__("uvicorn").__version__ + r"""
pydantic==""" + __import__("pydantic").__version__ + r"""
mlflow==""" + mlflow.__version__ + r"""
"""

with open("requirements.txt", "w") as f:
    f.write(pinned_requirements)

print(pinned_requirements)

# Deployment environment for the Cardiac Monitoring capstone (Week 9 Sprint 4)
# Frozen on Day 1 so Day 2's serving environment installs these exact versions.
pandas==3.0.2
numpy==2.4.4
scikit-learn==1.8.0
joblib==1.5.3
fastapi==0.141.1
uvicorn==0.46.0
pydantic==2.13.3
mlflow==3.16.0



> [!WARNING]
> **Pinned, not open-ended.** `scikit-learn==1.8.0` installs the exact version this model was serialized with.
> A bare `scikit-learn` (no version) could silently install a newer major version later and fail to unpickle
> `model.joblib` correctly -- a classic "worked on my machine" deployment bug this file exists specifically to
> prevent.

## 🧪 Hands-On Lab Recap: Sprint 4 Kickoff &amp; Serialization

| Step | Done |
|---|---|
| 1. Complete Sprint 4 planning and select the deployment backlog tasks | ✅ Section 1 |
| 2. Serialize the trained model and every preprocessing object to disk | ✅ Section 4 |
| 3. Write a script that loads them back and reproduces a known prediction | ✅ Section 5 |
| 4. Freeze a clean, pinned `requirements.txt` for the deployment environment | ✅ Section 6 |

## 📦 Artifacts Produced Today

- `model.joblib` — the trained Logistic Regression classifier
- `preprocessor.joblib` — the fitted `ColumnTransformer` (scaling + encoding)
- `requirements.txt` — pinned dependency versions for the deployment environment
- An MLflow run logging this model's parameters, metrics, and artifact

**Tomorrow (Day 2):** these two `.joblib` files get loaded into a FastAPI app behind a `/predict` endpoint, with
Pydantic validating every incoming request before it ever reaches the model.

## 🧰 Tools Used

![joblib](https://img.shields.io/badge/joblib-0B1F3A?style=flat-square)
![scikit-learn](https://img.shields.io/badge/Scikit--learn-1B4F8C?style=flat-square)
![MLflow](https://img.shields.io/badge/MLflow-0096C7?style=flat-square)
![pandas](https://img.shields.io/badge/pandas-0B1F3A?style=flat-square)
![Jupyter](https://img.shields.io/badge/Jupyter%20%2F%20Colab-0B1F3A?style=flat-square)
![Git](https://img.shields.io/badge/Git%20%26%20GitHub-0B1F3A?style=flat-square)

---

<p align="center"><sub>Week 9 · Sprint 4 · Day 1 of 5 → <b>Day 2: Serving the Model with FastAPI</b></sub></p>
